# Few-shot Universal Dependency Parsing with LLMs (Colab, models from Hugging Face)

Baseline experiment evaluating pretrained LLMs **without weight updates** across $N \in \{0, 3, 5, 10\}$ few-shot demonstrations, using full CoNLL-U-style tabular output (`ID \t FORM \t LEMMA \t UPOS \t XPOS \t FEATS \t HEAD \t DEPREL`) under gold tokenization.

**Colab setup**
1. Runtime → Change runtime type → GPU (A100/L4 recommended; Bolmo is loaded unquantized and needs about 16 GB+ for weights).
2. Add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar, enable notebook access). Llama and Aya are gated: accept their licenses on the model pages first.
3. Results are written to Google Drive, so the run is resumable across sessions.

**Metrics (all 9 columns from UDPipe 2.0 Table 5):** `UPOS`, `XPOS`, `UFeats`, `AllTags`, `Lemmas`, `UAS`, `LAS` (subtypes stripped), `MLAS`, `BLEX` (CoNLL 2018 definitions).

## 0. Environment (Colab)

In [ ]:
!pip install -q conllu "transformers>=4.57" accelerate bitsandbytes sentencepiece protobuf pandas
# Bolmo (mLSTM local encoder/decoder) needs these; Colab's Linux+GPU has Triton, so the triton kernels are registered.
# If the Bolmo model card lists different requirements, follow the model card.
!pip install -q xlstm mlstm_kernels

In [ ]:
import os, torch
from google.colab import drive

drive.mount("/content/drive")

# Hugging Face auth (Colab secret HF_TOKEN)
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception as e:
    print("No HF_TOKEN secret found; gated models (Llama, Aya) will fail to download.", e)

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory / 1e9:.1f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}")

## 1. Setup & Configuration

In [ ]:
import os, re, json, random, glob, time, gc, urllib.request
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, conllu

# Optional: cache HF downloads on Drive to skip re-downloading each session (needs ~15 GB free per 8B model).
USE_DRIVE_HF_CACHE = False
if USE_DRIVE_HF_CACHE:
    os.environ["HF_HOME"] = "/content/drive/MyDrive/nlp_udlm/hf_cache"

# ---------------- CONFIG ----------------
CFG = dict(
    models = {
        "llama":          "meta-llama/Llama-3.1-8B",
        "llama-instruct": "meta-llama/Meta-Llama-3-8B-Instruct",
        "aya":            "CohereLabs/aya-expanse-8b",
        "bolmo":          "allenai/Bolmo-7B",
    },
    # per-model loading options; quant=False loads in bf16/fp16 (used for Bolmo: custom architecture, 4-bit is unreliable)
    model_opts = {
        "llama":          dict(quant=True),
        "llama-instruct": dict(quant=True),
        "aya":            dict(quant=True),
        "bolmo":          dict(quant=False),
    },
    run_models     = ["llama", "llama-instruct", "aya", "bolmo"],
    treebanks      = ["en_ewt", "de_gsd", "af_afribooms", "fo_oft", "bar_maibaam", "hy_armtdp", "fa_perdt", "sq_staf"],
    shots          = [0, 3],          # extend to [0, 3, 5, 10] for the full baseline
    seeds          = [0],             # demonstration seeds (0-shot runs once with seed 0)
    n_eval         = 200,             # evaluation sentences per treebank (fixed random subset)
    max_eval_words = 40,
    max_demo_words = 25,
    batch_size     = 8,
    demo_source    = {},              # e.g. {"bar_maibaam": "de_gsd"} for cross-lingual demos
    held_out_demos = 50,              # demo pool carved from test when train/dev are absent
    strip_subtypes = True,
    out_dir        = "/content/drive/MyDrive/nlp_udlm/results",
)

LEVEL  = dict(en_ewt="HR", en_gum="HR", de_gsd="HR", de_hdt="HR",
              af_afribooms="MR", hy_armtdp="MR", fa_perdt="MR",
              fo_farpahc="LR", fo_oft="LR", bar_maibaam="LR", sq_staf="LR")
WEIGHT = dict(HR=1.0, MR=1.5, LR=2.0)

TREEBANKS = {
    "en_ewt":       ("UD_English-EWT", "en_ewt"),
    "en_gum":       ("UD_English-GUM", "en_gum"),
    "de_gsd":       ("UD_German-GSD", "de_gsd"),
    "de_hdt":       ("UD_German-HDT", "de_hdt"),
    "af_afribooms": ("UD_Afrikaans-AfriBooms", "af_afribooms"),
    "fo_farpahc":   ("UD_Faroese-FarPaHC", "fo_farpahc"),
    "fo_oft":       ("UD_Faroese-OFT", "fo_oft"),
    "bar_maibaam":  ("UD_Bavarian-MaiBaam", "bar_maibaam"),
    "hy_armtdp":    ("UD_Armenian-ArmTDP", "hy_armtdp"),
    "sq_staf":      ("UD_Albanian-STAF", "sq_staf"),
    "fa_perdt":     ("UD_Persian-PerDT", "fa_perdt"),
}

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CFG["out_dir"], exist_ok=True)

## 2. Data Loading

In [ ]:
def fetch(repo, prefix, split):
    path = f"{DATA_DIR}/{prefix}-ud-{split}.conllu"
    if not (os.path.exists(path) and os.path.getsize(path) > 0):
        url = f"https://raw.githubusercontent.com/UniversalDependencies/{repo}/master/{prefix}-ud-{split}.conllu"
        try:
            urllib.request.urlretrieve(url, path)
        except Exception:
            if os.path.exists(path):
                os.remove(path)
            return None
    return path

def clean(val):
    if val is None or val == "":
        return "_"
    return str(val).replace(" ", "_").replace("\t", "_").replace("\n", "")

def format_feats(feats):
    if not feats:
        return "_"
    if isinstance(feats, str):
        return clean(feats)
    return "|".join(f"{k}={v}" for k, v in sorted(feats.items()))

def read_conllu(path):
    sents = []
    with open(path, encoding="utf8") as f:
        for tl in conllu.parse_incr(f):
            toks = [t for t in tl if isinstance(t["id"], int)]
            if not toks or [t["id"] for t in toks] != list(range(1, len(toks) + 1)):
                continue
            heads = [t["head"] for t in toks]
            if any(h is None for h in heads) or sum(h == 0 for h in heads) != 1:
                continue
            sents.append(dict(
                id=tl.metadata.get("sent_id"),
                forms=[clean(t["form"]) for t in toks],
                lemmas=[clean(t.get("lemma")) for t in toks],
                upos=[clean(t.get("upos") or t.get("upostag")) for t in toks],
                xpos=[clean(t.get("xpos") or t.get("xpostag")) for t in toks],
                feats=[format_feats(t.get("feats")) for t in toks],
                heads=heads,
                deprels=[clean(t["deprel"]) for t in toks],
            ))
    return sents

DATA = {}
def load_treebank(name):
    repo, prefix = TREEBANKS[name]
    paths = {s: fetch(repo, prefix, s) for s in ["train", "dev", "test"]}
    test = read_conllu(paths["test"])
    if paths["train"]:
        pool, src = read_conllu(paths["train"]), "train"
    elif paths["dev"]:
        pool, src = read_conllu(paths["dev"]), "dev"
    else:
        rng = random.Random(123)
        rng.shuffle(test)
        k = CFG["held_out_demos"]
        pool, test, src = test[:k], test[k:], "test-heldout"
    pool = [s for s in pool if len(s["forms"]) <= CFG["max_demo_words"]]
    ev = [s for s in test if len(s["forms"]) <= CFG["max_eval_words"]]
    random.Random(123).shuffle(ev)
    ev = ev[:CFG["n_eval"]]
    DATA[name] = dict(pool=pool, eval=ev, pool_src=src)
    print(f"{name:14s} pool={len(pool):6d} ({src})  eval={len(ev)}")

for tb in set(CFG["treebanks"]) | set(CFG["demo_source"].values()):
    load_treebank(tb)

## 3. CoNLL-U Tabular Format, Parsing & UDPipe 2.0 Table 5 Scoring

- `UPOS`, `XPOS`, `UFeats`, `AllTags` (joint exact match of UPOS + XPOS + UFeats), `Lemmas`
- `UAS`, `LAS` (base UD relations without subtypes)
- `MLAS`: content words with correct head, base deprel, UPOS, UFeats, and identical `(deprel, UPOS, UFeats)` sets for functional children `{aux, cop, mark, det, clf, case}`
- `BLEX`: content words with correct head, base deprel, and lemma.

In [ ]:
METRICS_T5 = ["UPOS", "XPOS", "UFeats", "AllTags", "Lemmas", "UAS", "LAS", "MLAS", "BLEX"]

FUNC_RELS = {"aux", "cop", "mark", "det", "clf", "case"}
NON_CONTENT_RELS = FUNC_RELS | {"punct", "goeswith", "reparandum", "cc", "flat", "compound", "list", "orphan", "fixed"}

def base(label):
    return label.split(":")[0]

def norm_feats(f):
    if not f or f == "_":
        return "_"
    parts = sorted(p.strip() for p in f.split("|") if p.strip() and p.strip() != "_")
    return "|".join(parts) if parts else "_"

def table_of(s, strip=True):
    rows = []
    for i in range(len(s["forms"])):
        rel = (base(s["deprels"][i]) if strip else s["deprels"][i]).lower()
        rows.append(
            f"{i+1}\t{s['forms'][i]}\t{s['lemmas'][i]}\t{s['upos'][i]}\t"
            f"{s['xpos'][i]}\t{s['feats'][i]}\t{s['heads'][i]}\t{rel}"
        )
    return "\n".join(rows)

def parse_table(txt, n):
    out = dict(
        lemmas=[None] * n,
        upos=[None] * n,
        xpos=[None] * n,
        feats=[None] * n,
        heads=[None] * n,
        labs=[None] * n,
    )
    for line in txt.split("\n"):
        p = line.split("\t")
        if len(p) < 8:
            p = line.split()
            if len(p) < 8:
                continue
        if not p[0].strip().isdigit():
            continue
        i = int(p[0])
        h = p[6].strip()
        if 1 <= i <= n and out["heads"][i - 1] is None and h.isdigit():
            out["lemmas"][i - 1] = p[2].strip()
            out["upos"][i - 1]   = p[3].strip().upper()
            out["xpos"][i - 1]   = p[4].strip()
            out["feats"][i - 1]  = norm_feats(p[5].strip())
            out["heads"][i - 1]  = int(h)
            out["labs"][i - 1]   = p[7].strip().lower()
    return out

def _func_children(heads, rels, upos, feats):
    n = len(heads)
    fc = [[] for _ in range(n)]
    for j in range(n):
        h = heads[j]
        if h is not None and 1 <= h <= n and rels[j] in FUNC_RELS:
            fc[h - 1].append((rels[j], upos[j], feats[j]))
    return [sorted(x) for x in fc]

def evaluate(s, pred, strip=True):
    n = len(s["forms"])
    p = parse_table(pred, n)
    parsed = sum(h is not None for h in p["heads"])
    f = base if strip else (lambda x: x)

    g_upos  = [u.upper() for u in s["upos"]]
    g_xpos  = s["xpos"]
    g_feats = [norm_feats(ft) for ft in s["feats"]]
    g_lemma = s["lemmas"]
    g_heads = s["heads"]
    g_rels  = [f(d).lower() for d in s["deprels"]]

    p_rels  = [f(r).lower() if r is not None else None for r in p["labs"]]

    g_fc = _func_children(g_heads, g_rels, g_upos, g_feats)
    p_fc = _func_children(p["heads"], p_rels, p["upos"], p["feats"]) if parsed > 0 else [[] for _ in range(n)]

    c_upos = c_xpos = c_feats = c_alltags = c_lemma = c_head = c_las = 0
    n_content = c_mlas = c_blex = 0

    for j in range(n):
        is_content = g_rels[j] not in NON_CONTENT_RELS
        if is_content:
            n_content += 1
        if parsed == 0 or p["heads"][j] is None:
            continue

        u_ok = p["upos"][j] == g_upos[j]
        x_ok = p["xpos"][j] == g_xpos[j]
        f_ok = p["feats"][j] == g_feats[j]
        l_ok = p["lemmas"][j] == g_lemma[j]
        h_ok = p["heads"][j] == g_heads[j]
        las_ok = h_ok and (p_rels[j] == g_rels[j])

        c_upos    += int(u_ok)
        c_xpos    += int(x_ok)
        c_feats   += int(f_ok)
        c_alltags += int(u_ok and x_ok and f_ok)
        c_lemma   += int(l_ok)
        c_head    += int(h_ok)
        c_las     += int(las_ok)

        if is_content and las_ok:
            if u_ok and f_ok and p_fc[j] == g_fc[j]:
                c_mlas += 1
            if l_ok:
                c_blex += 1

    return dict(
        n_gold=n, n_content=n_content, valid=parsed > 0,
        c_upos=c_upos, c_xpos=c_xpos, c_feats=c_feats, c_alltags=c_alltags,
        c_lemma=c_lemma, c_head=c_head, c_las=c_las, c_mlas=c_mlas, c_blex=c_blex,
    )

def summarize(rs):
    g  = sum(r["n_gold"] for r in rs)
    gc = sum(r["n_content"] for r in rs)
    return dict(
        UPOS    = 100 * sum(r["c_upos"]    for r in rs) / g,
        XPOS    = 100 * sum(r["c_xpos"]    for r in rs) / g,
        UFeats  = 100 * sum(r["c_feats"]   for r in rs) / g,
        AllTags = 100 * sum(r["c_alltags"] for r in rs) / g,
        Lemmas  = 100 * sum(r["c_lemma"]   for r in rs) / g,
        UAS     = 100 * sum(r["c_head"]    for r in rs) / g,
        LAS     = 100 * sum(r["c_las"]     for r in rs) / g,
        MLAS    = 100 * sum(r["c_mlas"]    for r in rs) / max(gc, 1),
        BLEX    = 100 * sum(r["c_blex"]    for r in rs) / max(gc, 1),
    )

# Offline oracle verification (all 9 metrics must score 100.0)
for tb in CFG["treebanks"]:
    orc = summarize([evaluate(x, table_of(x, CFG["strip_subtypes"]), CFG["strip_subtypes"]) for x in DATA[tb]["eval"]])
    for m_name, val in orc.items():
        assert abs(val - 100.0) < 1e-6, (tb, m_name, val)
print("Oracle check passed (all 9 metrics = 100.0 across all treebanks).")

## 4. Prompts, Model Loading & Generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LANG = dict(en="English", de="German", af="Afrikaans", fo="Faroese", bar="Bavarian", hy="Armenian", fa="Persian", sq="Albanian")
def lang_of(tb):
    return LANG.get(tb.split("_")[0], tb.split("_")[0])

INSTR_TAB = (
    "Universal Dependencies morphological analysis, lemmatization, and dependency parsing of {lang} sentences. "
    "For each sentence, write a table in CoNLL-U style, one word per line, with eight tab-separated columns:\n"
    "ID\tFORM\tLEMMA\tUPOS\tXPOS\tFEATS\tHEAD\tDEPREL\n"
    "Use '_' when XPOS or FEATS is absent. HEAD is the ID of the word's syntactic head (0 for the root word, "
    "whose DEPREL is root). DEPREL is a Universal Dependencies relation without subtypes. "
    "Exactly one word has HEAD 0.\n\n"
)

def build_prompt(demos, s, lang=""):
    p = INSTR_TAB.replace("{lang}", lang)
    for d in demos:
        p += f"Sentence: {' '.join(d['forms'])}\n{table_of(d, strip=CFG['strip_subtypes'])}\n\n"
    return p + f"Sentence: {' '.join(s['forms'])}\n"

def get_demos(tb_eval, N, seed):
    src = CFG["demo_source"].get(tb_eval, tb_eval)
    pool = list(DATA[src]["pool"])
    random.Random(f"{src}-{seed}").shuffle(pool)
    return pool[:N]

MODEL = TOK_ = None

def free_gpu():
    global MODEL, TOK_
    MODEL = TOK_ = None
    gc.collect()
    torch.cuda.empty_cache()

def load_model(m):
    """Load model key `m` from the Hugging Face Hub. Returns the usable context length."""
    global MODEL, TOK_
    free_gpu()
    hf_id = CFG["models"][m]
    opts = CFG["model_opts"].get(m, dict(quant=True))

    dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    kw = dict(device_map={"": 0}, trust_remote_code=True, dtype=dt)
    if opts["quant"]:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dt)

    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(hf_id, **kw)
    model.eval()
    MODEL, TOK_ = model, tok
    ctx = getattr(model.config, "max_position_embeddings", None) or 4096
    ctx = min(ctx, 16384)  # cap: keeps prompt budgeting sane on a single Colab GPU
    print(f"{hf_id} loaded (quant={opts['quant']}); usable context = {ctx}")
    return ctx

def fit_demos(demos, s, ctx, reserve, lang=""):
    while True:
        p = build_prompt(demos, s, lang)
        if len(TOK_(p).input_ids) + reserve <= ctx or not demos:
            return demos, p
        demos = demos[:-1]

@torch.no_grad()
def generate(prompts, max_new):
    enc = TOK_(prompts, return_tensors="pt", padding=True).to(MODEL.device)
    kw = dict(max_new_tokens=max_new, do_sample=False, pad_token_id=TOK_.pad_token_id)
    try:
        out = MODEL.generate(**enc, stop_strings=["\nSentence:", "\n\nSentence:"], tokenizer=TOK_, **kw)
    except Exception:
        out = MODEL.generate(**enc, **kw)
    texts = TOK_.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return [t.lstrip("\n").split("\n\n")[0].split("Sentence:")[0].rstrip() for t in texts]

def run_batch(batch, prompts):
    reserve = max(len(TOK_(table_of(s)).input_ids) for s in batch)
    max_new = min(int(reserve * 1.4) + 24, 3072)
    try:
        return generate(prompts, max_new)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return [generate([p], max_new)[0] for p in prompts]

### 4b. Smoke test (optional, recommended before a full Bolmo run)
Loads one model, parses one sentence with 0 shots, and prints the raw output. If Bolmo fails here, you find out in a minute instead of mid-run.

In [ ]:
SMOKE_MODEL = "bolmo"   # any key from CFG["models"]
ctx = load_model(SMOKE_MODEL)
tb = CFG["treebanks"][0]
s = DATA[tb]["eval"][0]
p = build_prompt(get_demos(tb, 3, 0), s, lang_of(tb))
print(generate([p], 300)[0])
print("--- gold ---")
print(table_of(s))
free_gpu()

## 5. Experiment Loop (Resumable, results on Drive)

In [ ]:
def run_key(m, tb, N, seed):
    return f"{m}__{tb}__{CFG['demo_source'].get(tb, tb)}__N{N}__s{seed}"

def run_all():
    os.makedirs(CFG["out_dir"], exist_ok=True)
    for m in CFG["run_models"]:
        todo = [(tb, N, sd) for tb in CFG["treebanks"] for N in CFG["shots"]
                for sd in (CFG["seeds"] if N > 0 else [0])
                if not os.path.exists(f"{CFG['out_dir']}/{run_key(m, tb, N, sd)}.json")]
        if not todo:
            print(m, ": all done")
            continue
        try:
            ctx = load_model(m)
        except Exception as e:
            print(f"!! Could not load {m}: {type(e).__name__}: {str(e)[:300]}\n   Skipping.")
            free_gpu()
            continue
        for tb, N, sd in todo:
            t0 = time.time()
            ev = DATA[tb]["eval"]
            demos0 = get_demos(tb, N, sd)
            items = []
            for s in ev:
                reserve = int(len(TOK_(table_of(s)).input_ids) * 1.4) + 24
                d, p = fit_demos(demos0, s, ctx, reserve=reserve, lang=lang_of(tb))
                items.append((s, p))
            order = sorted(range(len(items)), key=lambda i: len(items[i][1]))
            preds = [None] * len(items)
            for b in range(0, len(order), CFG["batch_size"]):
                idx = order[b:b + CFG["batch_size"]]
                outs = run_batch([items[i][0] for i in idx], [items[i][1] for i in idx])
                for i, o in zip(idx, outs):
                    preds[i] = o
            res = [evaluate(s, pr, CFG["strip_subtypes"]) for (s, _), pr in zip(items, preds)]
            summ = summarize(res)
            with open(f"{CFG['out_dir']}/{run_key(m, tb, N, sd)}.json", "w") as out_f:
                json.dump(dict(model=m, treebank=tb, N=N, seed=sd, summary=summ,
                               preds=[dict(id=s["id"], pred=pr) for (s, _), pr in zip(items, preds)]), out_f)
            print(f"{m:14s} {tb:13s} N={N:2d} s={sd}  "
                  f"UPOS={summ['UPOS']:5.2f} Lem={summ['Lemmas']:5.2f} "
                  f"UAS={summ['UAS']:5.2f} LAS={summ['LAS']:5.2f} "
                  f"MLAS={summ['MLAS']:5.2f} BLEX={summ['BLEX']:5.2f}  ({time.time()-t0:.0f}s)")
        free_gpu()

run_all()

## 6. Results Summary

In [ ]:
files = sorted(glob.glob(f"{CFG['out_dir']}/*.json"))
if not files:
    print(f"No result files found in {CFG['out_dir']}. Run Section 5 first.")
else:
    rows = []
    for f in files:
        j = json.load(open(f))
        row = dict(model=j["model"], treebank=j["treebank"], N=j["N"], seed=j["seed"])
        row.update({k: j["summary"][k] for k in METRICS_T5})
        rows.append(row)
    df = pd.DataFrame(rows)

    tb_mean = df.groupby(["model", "N", "treebank"])[METRICS_T5].mean().reset_index()
    macro = tb_mean.groupby(["model", "N"])[METRICS_T5].mean().round(2)

    print("=== Macro-Average Across Treebanks (UDPipe 2.0 Table 5 Metrics) ===")
    display(macro)

    for (m, N), g in tb_mean.groupby(["model", "N"]):
        tab = g.set_index("treebank")[METRICS_T5].copy()
        tab.loc["macro-avg"] = tab.mean()
        print(f"\n=== Model: {m} | N={N} shots ===")
        display(tab.round(2))